# NB-03_revision_tablas_dcv

Process Flow SAS: **Revisión tablas DCV** — `PFD-qSlM5lap8p2JfvZN`

In [ ]:
# ========= Celda 1: Configuración =========
import pandas as pd
import numpy as np
import os
import sqlalchemy
import datetime
from pathlib import Path

# Conexión a BD — editable acá; SASMIG_DB_URL (orquestador) tiene
# prioridad si está definida (SUPUESTO: verificar servidor y base
# antes de correr contra datos reales).
config_db = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=PLATDAT,1433;"
    "DATABASE=GOBGENER;"
    "Authentication=ActiveDirectoryIntegrated;"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "MARS_Connection=Yes;"
)
engine = sqlalchemy.create_engine(
    os.environ.get("SASMIG_DB_URL", f"mssql+pyodbc:///?odbc_connect={config_db}"),
    pool_pre_ping=True,
    fast_executemany=True,
)

# Logging liviano de resultados — aprobado en la entrevista (Fase 4)
_LOG_PATH = Path("log") / "NB-03_revision_tablas_dcv.log"
_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
def _log(label, value=None):
    """Una línea por celda: imprime y persiste. Jamás rompe la corrida."""
    try:
        if hasattr(value, "shape"):
            detail = f"{value.shape[0]} filas x {value.shape[1]} cols"
        elif isinstance(value, int):
            detail = f"{value} filas"
        elif value is None:
            detail = "ok"
        else:
            detail = str(value)
        line = f"[{datetime.datetime.now():%Y-%m-%d %H:%M:%S}] {label}: {detail}"
        print(line)
        with open(_LOG_PATH, "a", encoding="utf-8") as fh:
            fh.write(line + "\n")
    except Exception:
        pass  # el log nunca puede tumbar el notebook
with open(_LOG_PATH, "a", encoding="utf-8") as _fh:
    _fh.write(f"\n=== corrida {datetime.datetime.now():%Y-%m-%d %H:%M:%S} ===\n")


## S1_01_RevDCV

Audita la consistencia contable Balance Final = Balance Inicio para 9 tablas de posiciones DCV (certificados, activos/pasivos de fondos de bonos, offshore, HC, oficinas, auxiliares y CCAF), invirtiendo signo salvo en el Balance Final y reportando diferencias mayores a 0.05 por año/trimestre/sector/instrumento

*confianza: low · verificador: approve · SAS: PROC SQL CREATE TABLE leyendo archivos .sas7bdat por ruta absoluta + PROC TABULATE de diagnostico, repetido 9 veces sobre distintos archivos*

In [ ]:
# ========= S1_01_RevDCV =========
raise NotImplementedError(
    "Nodo S1_01_RevDCV: auditoria repetitiva (9 tablas SAS7BDAT) que lee directamente "
    "archivos .sas7bdat por ruta absoluta hardcodeada "
    "'/sasdata/BCCH/GEM_DCNI/02_CNSI/05_DCV/Data/DCVRES/*.sas7bdat' via PROC SQL "
    "'FROM <ruta>.sas7bdat' -- este patron de lectura NO esta en la tabla de "
    "patrones (no es PROC IMPORT ni tabla de BD del catalogo de conexiones: "
    "GOBGENER/TABLAS no incluyen estos archivos DCVRES). Requiere pyreadstat.read_sas7bdat "
    "sobre una ruta relativa del workspace (M-001) que el proyecto aun no ha definido "
    "para estos 9 archivos: t_aj_cd_321_6_AF32, t_aj_dcv_af31_act_c21, t_aj_dcv_af31_pas, "
    "t_aj_dcv_af32_act_ao, t_aj_dcv_af32_act_hc, t_aj_dcv_af32_act_ofis, t_aj_dcv_af32_pas_ao, "
    "t_aj_dcv_af32_pas_hc, t_aj_dcv_aux_oif_af32, t_aj_dcv_ccaf_af32_pas. Ademas el nodo "
    "solo produce salidas PROC TABULATE (reportes de consistencia BF=BI, sin escritura a "
    "tablas de salida persistentes: WORK.T1/P_BF_T1/P_BI_T1/P_DIF_T1 se recrean y se "
    "DROPean en cada bloque, son de solo diagnostico). Falta: (1) la ruta relativa real "
    "donde deben vivir estos 9 .sas7bdat en el workspace del proyecto, (2) confirmar si "
    "los reportes PROC TABULATE deben materializarse como DataFrames de salida o solo "
    "como impresion/logging de diagnostico."
)

## Program

Extrae del archivo consolidado AF3 los registros correspondientes al año 2025

*confianza: medium · verificador: approve · SAS: PROC SQL CREATE TABLE AS SELECT * FROM archivo SAS7BDAT con filtro WHERE*

In [ ]:
# ========= Program =========
# Ruta absoluta reemplazada por ruta relativa al workspace (M-001)
ruta_base_af3 = Path("sasdata") / "BCCH" / "GEM_DCNI" / "02_CNSI" / "05_DCV" / "Data" / "DCVRES" / "base_af3_total_v2.sas7bdat"
fff_full = pd.read_sas(ruta_base_af3, encoding="latin1")
# filtro año=2025 (verificar tipo de la columna 'año' tras la lectura binaria SAS)
fff = fff_full[fff_full["año"] == 2025].copy()
_log("fff", fff)
